In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
SIMULACIÓN ELECTORAL COSTA RICA 2026 - VERSIÓN FINAL INTEGRADA
================================================================================
Combina:
1. Modelado bayesiano WLS con logits (nivel profesional)
2. Datos históricos de abstencionismo CR (1982-2024)
3. Patrones de variabilidad electoral (votantes ocasionales)
4. 12 encuestas octubre 2025 - enero 2026
5. House effects + incertidumbre paramétrica

Autor: Agustín Gómez (CIOdD-UCR)
Fecha: Enero 2026
Fuentes:
- Encuestas nacionales (CIEP, IDESPO, OPOL, Demoscopia, CID Gallup)
- "Abstencionistas en Costa Rica" (Raventós et al., 2005)
- "Deshojar la margarita en las urnas" (Alfaro Redondo, PEN 2024)
================================================================================
"""

from __future__ import annotations

import re
import math
from dataclasses import dataclass
from datetime import date
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# =============================================================================
# CONFIGURACIÓN GLOBAL
# =============================================================================
ELECTION_DATE = pd.Timestamp("2026-02-01")
SEED = 42
N_SIMS = 200_000

# Datos históricos de participación CR
HISTORICO_PARTICIPACION = {
    1982: 78.6, 1986: 81.8, 1990: 81.8, 1994: 81.1,
    1998: 70.0,  # CAMBIO ESTRUCTURAL
    2002: 68.8, 2006: 64.8, 2010: 68.8, 2014: 68.2,
    2018: 65.7, 2022: 60.0
}

# Patrones de variabilidad electoral (PEN 2024)
VOTANTES_HABITUALES = 0.31  # Bajó de 47% (1994) a 31% (2022)
VOTANTES_OCASIONALES = 0.44  # Aumentó de 38% a 44%
ABSTIENEN_SIEMPRE = 0.01     # Solo 1% nunca vota
BAJA_PROPENSION = 0.24       # 25% se abstuvo en 4+ elecciones

# =============================================================================
# DATOS DE ENCUESTAS (OCTUBRE 2025 - ENERO 2026)
# =============================================================================
POLL_COLS = [
    "ciep_ucr_1", "idespo_una_1", "opol_1", "demoscopia_1",
    "opol_2", "opol_3", "ciep_ucr_2", "idespo_una_2",
    "opol_4", "demoscopia_2", "opol_5", "cid_gallup"
]

RAW_ROWS: List[Tuple[str, str, List]] = [
    ("MARGEN DE ERROR", "Margen",
     [2.7, 3.3, 2.2, 2.83, 2.16, 2.1, 2.3, 3.3, 2.24, 2.83, 2.11, 2.87]),

    ("MODALIDAD/MUESTRA", "Modalidad",
     ["Telefónica celular", "Telefónica celular", "Presencial", "Presencial",
      "Presencial", "Presencial", "Telefónica", "Telefónica celular",
      "Presencial", "Presencial", "Presencial", "Presencial"]),

    ("FECHAS TRABAJO DE CAMPO", "Fechas",
     ["8 al 15 de octubre", "16 al 25 de octubre", "25 al 27 de octubre",
      "13 al 31 de octubre", "7 al 10 de noviembre", "20 al 24 de noviembre",
      "19 al 28 de noviembre", "23 al 29 de noviembre", "5 al 8 de diciembre",
      "6 al 13 de diciembre", "19 al 22 de diciembre", "29 dic-5 de enero"]),

    ("FECHA PUBLICACIÓN", "Publicación",
     ["22 de octubre", "6 de noviembre", "29 de octubre", "13 de noviembre",
      "12 de noviembre", "25 de noviembre", "3 de diciembre", "8 de diciembre",
      "10 de diciembre", "16 de diciembre", "23 de diciembre", "6 de enero"]),

    ("PREGUNTA INTENCIÓN DE VOTO", "Pregunta",
     ["Abierta", "Abierta", "Papeleta Simulada", "Abierta",
      "Papeleta Simulada", "Papeleta Simulada", "Papeleta Simulada", "Panel-Abierta",
      "Abierta", "Papeleta Simulada", "Papeleta Simulada", "Lista"]),

    # VOTO (corregido a 12 columnas)
    ("INDECISOS", "Voto", [55.00, 52.40, 38.79, 56.70, 42.30, 37.37, 34.14, 45.00, 43.90, 32.67, 41.70, 33.77]),
    ("LAURA FERNÁNDEZ", "Voto", [25.00, 28.10, 31.20, 21.40, 21.00, 37.85, 37.66, 30.00, 32.80, 38.01, 27.40, 39.45]),
    ("ÁLVARO RAMOS", "Voto", [7.00, 6.20, 7.41, 9.00, 10.40, 7.19, 6.91, 8.00, 6.60, 6.12, 11.30, 5.64]),
    ("FABRICIO ALVARADO", "Voto", [np.nan, np.nan, 4.84, 3.70, 6.50, 3.97, 3.66, 1.00, np.nan, 3.81, 3.60, 2.98]),
    ("ARIEL ROBLES", "Voto", [3.00, 2.30, 3.72, 2.10, 4.10, 2.35, 3.38, 5.00, 3.70, 3.63, 4.80, 3.13]),
    ("JUAN CARLOS HIDALGO", "Voto", [np.nan, 1.20, 2.69, np.nan, 2.40, 2.21, 3.10, 1.00, np.nan, 2.14, 2.00, 2.49]),
    ("NATALIA DÍAZ", "Voto", [np.nan, np.nan, 2.47, np.nan, np.nan, 1.76, 1.49, 1.00, np.nan, 3.05, np.nan, 2.84]),
    ("CLAUDIA DOBLES", "Voto", [3.00, 2.90, 2.92, 3.00, 2.90, 1.29, 2.80, 4.00, 5.20, 2.39, 3.10, 2.63]),
    ("ELI FEINZAIG", "Voto", [np.nan, np.nan, 1.68, np.nan, 2.70, 1.22, np.nan, 1.00, np.nan, np.nan, 3.10, np.nan]),
    ("FERNANDO ZAMORA", "Voto", [np.nan, np.nan, np.nan, np.nan, np.nan, 1.15, 2.01, np.nan, np.nan, 2.12, np.nan, 1.52]),
    ("ANA VIRGINIA CALZADA", "Voto", [np.nan, np.nan, np.nan, 2.50, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]),
    ("LUIS AMADOR", "Voto", [np.nan, np.nan, np.nan, 2.50, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]),
    ("OTROS", "Voto", [4.10, 3.30, 2.24, 4.10, 2.70, 2.11, 3.93, 1.50, 4.40, 3.91, 3.10, 2.64]),
    ("NO RESPONDE", "Voto", [np.nan, 1.60, 0.24, np.nan, np.nan, 0.50, np.nan, np.nan, 1.60, np.nan, 2.15, np.nan]),
    ("NULO/BLANCO", "Voto", [2.50, 2.00, 1.80, np.nan, np.nan, 1.03, 0.92, 2.60, 1.80, 2.15, np.nan, 2.91]),
]

# =============================================================================
# UTILIDADES
# =============================================================================
MONTH_MAP = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6,
    "julio": 7, "agosto": 8, "septiembre": 9, "setiembre": 9,
    "octubre": 10, "noviembre": 11, "diciembre": 12
}

def parse_publish_date(s: str) -> pd.Timestamp:
    """Parse '22 de octubre' -> Timestamp con año apropiado"""
    s = str(s).strip().lower()
    m = re.match(r"(\d{1,2})\s*de\s*([a-záéíóúñ]+)", s)
    if not m:
        return pd.NaT
    day = int(m.group(1))
    month = MONTH_MAP.get(m.group(2))
    if month is None:
        return pd.NaT
    year = 2025 if month in (10, 11, 12) else 2026
    return pd.Timestamp(date(year, month, day))

def qtype_norm(x: str) -> str:
    """Normaliza tipo de pregunta"""
    x = str(x).strip().lower()
    if "papeleta" in x: return "papeleta"
    if "lista" in x: return "lista"
    if "panel" in x: return "panel"
    if "abierta" in x: return "abierta"
    return "otro"

def logit(p: np.ndarray) -> np.ndarray:
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def expit(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))

@dataclass
class FittedModel:
    name: str
    beta: np.ndarray
    cov: np.ndarray
    sigma: float
    columns: List[str]
    house_sd: float

# =============================================================================
# CONSTRUCCIÓN DE DATOS
# =============================================================================
def build_dataframe() -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Construye DataFrame de encuestas"""
    # Construir datos fila por fila
    data_rows = []
    for candidato, tipo, vals in RAW_ROWS:
        row = [candidato, tipo] + vals
        data_rows.append(row)

    df_raw = pd.DataFrame(data_rows, columns=["candidato", "tipo"] + POLL_COLS)

    # Metadatos
    moe = df_raw.loc[df_raw["candidato"] == "MARGEN DE ERROR", POLL_COLS].iloc[0].to_dict()
    pub = df_raw.loc[df_raw["candidato"] == "FECHA PUBLICACIÓN", POLL_COLS].iloc[0].to_dict()
    qst = df_raw.loc[df_raw["candidato"] == "PREGUNTA INTENCIÓN DE VOTO", POLL_COLS].iloc[0].to_dict()

    pollster_map = {
        "ciep_ucr_1": "CIEP-UCR", "ciep_ucr_2": "CIEP-UCR",
        "idespo_una_1": "IDESPO-UNA", "idespo_una_2": "IDESPO-UNA",
        "opol_1": "OPOL", "opol_2": "OPOL", "opol_3": "OPOL",
        "opol_4": "OPOL", "opol_5": "OPOL",
        "demoscopia_1": "Demoscopia", "demoscopia_2": "Demoscopia",
        "cid_gallup": "CID Gallup"
    }

    polls = pd.DataFrame([
        {
            "poll_id": c,
            "pollster": pollster_map.get(c, c),
            "moe": float(moe[c]),
            "publish_raw": pub[c],
            "publish_date": parse_publish_date(pub[c]),
            "qtype": qtype_norm(qst[c])
        }
        for c in POLL_COLS
    ]).sort_values("publish_date").reset_index(drop=True)

    polls["n_eff"] = 0.25 * (1.96 * 100 / polls["moe"]) ** 2

    return df_raw, polls

def build_votes_wide(df_raw: pd.DataFrame, polls: pd.DataFrame) -> pd.DataFrame:
    """Pivotea datos de voto a formato ancho"""
    vote_rows = df_raw[df_raw["tipo"] == "Voto"].copy()

    long = vote_rows.melt(
        id_vars=["candidato", "tipo"],
        value_vars=POLL_COLS,
        var_name="poll_id",
        value_name="pct_total"
    ).merge(polls[["poll_id", "pollster", "publish_date", "qtype", "n_eff"]],
            on="poll_id", how="left")

    long["candidate"] = long["candidato"].str.strip().str.upper()

    wide = long.pivot_table(
        index=["poll_id", "pollster", "publish_date", "qtype", "n_eff"],
        columns="candidate",
        values="pct_total",
        aggfunc="first"
    ).reset_index()

    # Asegurar columnas críticas
    for col in ["INDECISOS", "NO RESPONDE", "NULO/BLANCO"]:
        if col not in wide.columns:
            wide[col] = 0.0

    wide["INDECISOS"] = wide["INDECISOS"].fillna(0.0)
    wide["NO RESPONDE"] = wide["NO RESPONDE"].fillna(0.0)
    wide["NULO/BLANCO"] = wide["NULO/BLANCO"].fillna(0.0)

    # Variables clave
    wide["U"] = wide["INDECISOS"] + wide["NO RESPONDE"]  # Indecisos totales
    wide["B"] = wide["NULO/BLANCO"]  # Blancos/nulos
    wide["D"] = (100.0 - wide["U"] - wide["B"]).clip(lower=1e-6)  # Decididos

    wide = wide.sort_values("publish_date").reset_index(drop=True)
    wide["t_days"] = (wide["publish_date"] - wide["publish_date"].min()).dt.days.astype(float)

    return wide

# =============================================================================
# MODELADO BAYESIANO WLS
# =============================================================================
def build_design(df: pd.DataFrame, baseline_qtype: str = "papeleta") -> Tuple[pd.DataFrame, List[str], List[str]]:
    """
    Construye matriz de diseño:
    X = intercept + t_days + qtype_dummies + pollster_dummies
    """
    X = pd.DataFrame({"t_days": df["t_days"].astype(float).values})

    # Dummies de tipo de pregunta (baseline = papeleta)
    qcats = [baseline_qtype] + [c for c in sorted(df["qtype"].unique()) if c != baseline_qtype]
    q = pd.Categorical(df["qtype"], categories=qcats, ordered=True)
    q_dum = pd.get_dummies(q, drop_first=True, prefix="q")

    # Dummies de encuestadora (baseline = primera alfabéticamente)
    pcats = sorted(df["pollster"].unique())
    p = pd.Categorical(df["pollster"], categories=pcats, ordered=True)
    p_dum = pd.get_dummies(p, drop_first=True, prefix="h")

    X = pd.concat([X, q_dum, p_dum], axis=1)
    X = sm.add_constant(X, has_constant="add").astype(float)

    return X, list(q_dum.columns), list(p_dum.columns)

def fit_wls_logit(df: pd.DataFrame, y_pct_col: str, w_col: str = "n_eff") -> FittedModel:
    """
    Ajusta modelo WLS sobre logit(y/100)
    """
    X, qdum_cols, hdum_cols = build_design(df)

    y_pct = df[y_pct_col].astype(float).values
    y = logit(y_pct / 100.0)
    w = df[w_col].astype(float).fillna(df[w_col].median()).values

    res = sm.WLS(y, X, weights=w).fit()
    beta = res.params.values
    cov = res.cov_params().values

    resid = y - res.fittedvalues
    sigma = float(np.sqrt(np.average(resid**2, weights=w)))

    # House effects
    house_effects = [0.0] + [float(res.params.get(col, 0.0)) for col in hdum_cols]
    house_sd = float(np.std(house_effects, ddof=1)) if len(house_effects) > 1 else 0.0

    return FittedModel(
        name=y_pct_col,
        beta=beta,
        cov=cov,
        sigma=sigma,
        columns=list(res.params.index),
        house_sd=house_sd
    )

def fit_wls_linear(df: pd.DataFrame, y_col: str, w_col: str = "n_eff") -> FittedModel:
    """
    Ajusta modelo WLS lineal (para logits de composición)
    """
    X, qdum_cols, hdum_cols = build_design(df)

    y = df[y_col].astype(float).values
    w = df[w_col].astype(float).fillna(df[w_col].median()).values

    res = sm.WLS(y, X, weights=w).fit()
    beta = res.params.values
    cov = res.cov_params().values

    resid = y - res.fittedvalues
    sigma = float(np.sqrt(np.average(resid**2, weights=w)))

    house_effects = [0.0] + [float(res.params.get(col, 0.0)) for col in hdum_cols]
    house_sd = float(np.std(house_effects, ddof=1)) if len(house_effects) > 1 else 0.0

    return FittedModel(
        name=y_col,
        beta=beta,
        cov=cov,
        sigma=sigma,
        columns=list(res.params.index),
        house_sd=house_sd
    )

def make_pred_vector(columns: List[str], t_days: float, qtype: str = "papeleta") -> np.ndarray:
    """Construye vector de predicción para fecha futura"""
    row = {c: 0.0 for c in columns}
    row["const"] = 1.0
    row["t_days"] = float(t_days)
    if qtype != "papeleta":
        qcol = f"q_{qtype}"
        if qcol in row:
            row[qcol] = 1.0
    return np.array([row[c] for c in columns], dtype=float)

# =============================================================================
# SIMULACIÓN MONTE CARLO CON DATOS HISTÓRICOS CR
# =============================================================================
def adjust_for_cr_historical_patterns(sims_basic: pd.DataFrame, rng) -> pd.DataFrame:
    """
    Ajusta simulaciones básicas con patrones históricos de CR:
    - Votantes ocasionales (44%) tienden a reincorporarse con liderazgo claro
    - Solo 1% nunca vota
    - 31% son votantes habituales consistentes
    """
    n = len(sims_basic)
    sims = sims_basic.copy()

    # Clasificar cada simulación según tipo de elector
    # CORREGIDO: probabilidades que sumen exactamente 1.0
    p_habitual = 0.31
    p_ocasional_vota = 0.44 * 0.70  # 30.8%
    p_ocasional_abstiene = 0.44 * 0.30  # 13.2%
    p_abstiene_siempre = 0.01
    # Total aproximado: 31 + 30.8 + 13.2 + 1 = 76%
    # Ajustar para que sume 1.0
    total_prob = p_habitual + p_ocasional_vota + p_ocasional_abstiene + p_abstiene_siempre

    # Renormalizar
    probs = np.array([p_habitual, p_ocasional_vota, p_ocasional_abstiene, p_abstiene_siempre])
    probs = probs / probs.sum()  # Asegurar suma = 1.0

    tipo_elector = rng.choice(
        ['habitual', 'ocasional_vota', 'ocasional_abstiene', 'abstiene_siempre'],
        size=n,
        p=probs
    )

    # Ajuste diferencial según tipo
    ajuste = np.zeros(n)

    # Votantes habituales: sin ajuste (ya están bien modelados)
    habitual_mask = (tipo_elector == 'habitual')

    # Ocasionales que votan: leve boost al líder (momentum)
    ocasional_vota_mask = (tipo_elector == 'ocasional_vota')
    ajuste[ocasional_vota_mask] = rng.normal(1.5, 0.5, size=ocasional_vota_mask.sum())

    # Ocasionales que se abstienen: reducción
    ocasional_abstiene_mask = (tipo_elector == 'ocasional_abstiene')
    ajuste[ocasional_abstiene_mask] = rng.normal(-2.0, 0.8, size=ocasional_abstiene_mask.sum())

    # Abstienen siempre: fuerte reducción
    abstiene_mask = (tipo_elector == 'abstiene_siempre')
    ajuste[abstiene_mask] = rng.normal(-5.0, 1.0, size=abstiene_mask.sum())

    # Aplicar ajuste a Laura (líder claro)
    sims["LAURA FERNÁNDEZ"] = (sims["LAURA FERNÁNDEZ"] + ajuste).clip(15, 75)

    # Renormalizar para mantener suma = 100
    included = ["LAURA FERNÁNDEZ", "ÁLVARO RAMOS", "FABRICIO ALVARADO",
                "ARIEL ROBLES", "CLAUDIA DOBLES", "OTROS", "RESTO"]
    total = sims[included].sum(axis=1)
    for c in included:
        sims[c] = 100.0 * sims[c] / total

    return sims

def run_simulation(df_wide: pd.DataFrame, n_sims: int = N_SIMS, seed: int = SEED) -> Dict:
    """
    Simulación Monte Carlo completa con:
    1. Modelado bayesiano WLS
    2. Ajustes por patrones históricos CR
    """
    print("\n" + "="*80)
    print("🎲 SIMULACIÓN MONTE CARLO - COSTA RICA 2026")
    print("="*80)

    # Filtrar encuestas con papeleta/lista (más fiables)
    df = df_wide[df_wide["qtype"].isin(["papeleta", "lista"])].copy().reset_index(drop=True)

    print(f"\nEncuestas usadas: {len(df)} (papeleta/lista)")
    print(f"Período: {df['publish_date'].min().date()} a {df['publish_date'].max().date()}")

    # Candidatos principales
    included = ["LAURA FERNÁNDEZ", "ÁLVARO RAMOS", "FABRICIO ALVARADO",
                "ARIEL ROBLES", "CLAUDIA DOBLES", "OTROS"]

    for c in included:
        if c not in df.columns:
            df[c] = 0.0
        df[c] = df[c].fillna(0.0)

    # Calcular RESTO (base para logits)
    p_included = df[included].sum(axis=1) / df["D"]
    p_base = (1.0 - p_included).clip(lower=1e-4)

    # z_k = log(p_k / p_resto)
    for c in included:
        p_k = (df[c] / df["D"]).clip(lower=1e-6)
        df[f"z_{c}"] = np.log(p_k / p_base)

    print("\n⏳ Ajustando modelos bayesianos WLS...")

    # Modelos para U (indecisos) y B (blancos/nulos)
    mU = fit_wls_logit(df, "U")
    mB = fit_wls_logit(df, "B")

    # Modelos para z_k (composición voto válido)
    mz = {c: fit_wls_linear(df, f"z_{c}") for c in included}

    print("✅ Modelos ajustados")

    # Predicción a fecha de elección
    t0 = df["publish_date"].min()
    t_e = float((ELECTION_DATE - t0).days)

    print(f"\n⏳ Generando {n_sims:,} simulaciones...")

    rng = np.random.default_rng(seed)
    sims = pd.DataFrame(index=np.arange(n_sims))

    # === FASE 1: Simular U y B ===
    for name, fm in [("U", mU), ("B", mB)]:
        x = make_pred_vector(fm.columns, t_e, qtype="papeleta")

        # Incertidumbre paramétrica
        beta_draw = rng.multivariate_normal(fm.beta, fm.cov, size=n_sims)
        eta = beta_draw @ x

        # Incertidumbre residual + house effect
        eta = eta + rng.normal(0, fm.sigma, size=n_sims) + rng.normal(0, fm.house_sd, size=n_sims)

        sims[name] = 100.0 * expit(eta)

    sims["U"] = sims["U"].clip(20, 70)  # Indecisos: rango plausible
    sims["B"] = sims["B"].clip(0, 10)   # Blancos/nulos: rango plausible
    sims["D"] = (100.0 - sims["U"] - sims["B"]).clip(1e-6, 100)

    # === FASE 2: Simular composición (z_k) ===
    Z = np.zeros((n_sims, len(included)))

    for j, c in enumerate(included):
        fm = mz[c]
        x = make_pred_vector(fm.columns, t_e, qtype="papeleta")

        beta_draw = rng.multivariate_normal(fm.beta, fm.cov, size=n_sims)
        mu = beta_draw @ x
        mu = mu + rng.normal(0, fm.sigma, size=n_sims) + rng.normal(0, fm.house_sd, size=n_sims)
        Z[:, j] = mu

    # Reconstruir shares: p_k = exp(z_k) / (1 + sum exp(z_k))
    expZ = np.exp(np.clip(Z, -20, 20))
    den = 1.0 + expZ.sum(axis=1)
    p_base_sim = 1.0 / den
    P = expZ * p_base_sim[:, None]

    for j, c in enumerate(included):
        sims[c] = 100.0 * P[:, j]

    sims["RESTO"] = 100.0 * p_base_sim

    print("✅ Simulación básica completada")

    # === FASE 3: Ajustar con patrones históricos CR ===
    print("\n⏳ Ajustando con patrones históricos de Costa Rica...")
    sims = adjust_for_cr_historical_patterns(sims, rng)

    print("✅ Ajustes aplicados")

    # === ANÁLISIS ===
    sims["LF"] = sims["LAURA FERNÁNDEZ"]
    sims["wins_round1"] = sims["LF"] >= 40.0

    # Top-2
    cand_all = included + ["RESTO"]
    vals = sims[cand_all].values
    top1_idx = np.argmax(vals, axis=1)
    top1 = np.array([cand_all[i] for i in top1_idx])
    vals2 = vals.copy()
    vals2[np.arange(n_sims), top1_idx] = -1
    top2_idx = np.argmax(vals2, axis=1)
    top2 = np.array([cand_all[i] for i in top2_idx])

    sims["top1"] = top1
    sims["top2"] = top2

    # Resumen
    out = {
        "polls_used": int(df.shape[0]),
        "summary": {
            "n_sims": int(n_sims),
            "election_date": str(ELECTION_DATE.date()),
            "p_LF_win_round1": float(sims["wins_round1"].mean()),
            "p_LF_top1": float((sims["top1"] == "LAURA FERNÁNDEZ").mean()),
            "LF_mean": float(sims["LF"].mean()),
            "LF_median": float(sims["LF"].median()),
            "LF_p5": float(sims["LF"].quantile(0.05)),
            "LF_p10": float(sims["LF"].quantile(0.10)),
            "LF_p90": float(sims["LF"].quantile(0.90)),
            "LF_p95": float(sims["LF"].quantile(0.95)),
            "U_mean": float(sims["U"].mean()),
            "B_mean": float(sims["B"].mean()),
        },
        "cand_summary": sims[cand_all].describe(percentiles=[0.05, 0.10, 0.50, 0.90, 0.95]).T,
        "top2_pairs": (sims.groupby(["top1", "top2"]).size() / n_sims).sort_values(ascending=False),
        "sims": sims,
        "historical_adjustments_applied": True,
    }

    return out

# =============================================================================
# VISUALIZACIÓN
# =============================================================================
def create_comprehensive_plots(out: Dict) -> None:
    """Crea visualizaciones profesionales"""
    sims = out["sims"]

    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle("SIMULACIÓN ELECTORAL COSTA RICA 2026 - PRIMERA RONDA\nIncorpora datos históricos + Modelado Bayesiano WLS",
                 fontsize=14, fontweight='bold')

    # 1. Distribución Laura Fernández
    ax = axes[0, 0]
    ax.hist(sims["LF"], bins=80, alpha=0.7, edgecolor='black')
    ax.axvline(40, color='red', linestyle='--', linewidth=2, label='Umbral 40%')
    ax.axvline(sims["LF"].mean(), color='blue', linestyle='-', linewidth=2, label=f'Media: {sims["LF"].mean():.1f}%')
    ax.axvline(sims["LF"].median(), color='green', linestyle='-', linewidth=2, label=f'Mediana: {sims["LF"].median():.1f}%')
    ax.set_xlabel("% del voto válido", fontsize=11)
    ax.set_ylabel("Frecuencia", fontsize=11)
    ax.set_title("Laura Fernández - Distribución simulada", fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

    # 2. Probabilidad acumulada
    ax = axes[0, 1]
    sorted_lf = np.sort(sims["LF"])
    cum_prob = np.arange(1, len(sorted_lf)+1) / len(sorted_lf)
    ax.plot(sorted_lf, cum_prob*100, linewidth=2)
    ax.axvline(40, color='red', linestyle='--', linewidth=2)
    ax.axhline(out["summary"]["p_LF_win_round1"]*100, color='green', linestyle='--',
               label=f'P(>40%) = {out["summary"]["p_LF_win_round1"]*100:.1f}%')
    ax.set_xlabel("% del voto válido", fontsize=11)
    ax.set_ylabel("Probabilidad acumulada (%)", fontsize=11)
    ax.set_title("Función de distribución acumulada", fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

    # 3. Indecisos (U)
    ax = axes[0, 2]
    ax.hist(sims["U"], bins=60, alpha=0.7, color='orange', edgecolor='black')
    ax.axvline(sims["U"].mean(), color='red', linestyle='--', linewidth=2,
               label=f'Media: {sims["U"].mean():.1f}%')
    ax.set_xlabel("% Indecisos + No responde", fontsize=11)
    ax.set_ylabel("Frecuencia", fontsize=11)
    ax.set_title("Distribución de Indecisos (U)", fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

    # 4. Boxplot top candidatos
    ax = axes[1, 0]
    top_cands = ["LAURA FERNÁNDEZ", "ÁLVARO RAMOS", "FABRICIO ALVARADO",
                 "ARIEL ROBLES", "CLAUDIA DOBLES"]
    data_box = [sims[c] for c in top_cands]
    bp = ax.boxplot(data_box, labels=[c.split()[0] for c in top_cands], patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
    ax.axhline(40, color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax.set_ylabel("% del voto válido", fontsize=11)
    ax.set_title("Distribución por candidato (Top 5)", fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

    # 5. Laura vs Segundo lugar
    ax = axes[1, 1]
    segundo = sims["ÁLVARO RAMOS"]
    ax.scatter(sims["LF"], segundo, alpha=0.05, s=1)
    ax.axhline(segundo.mean(), color='red', linestyle='--', alpha=0.7)
    ax.axvline(sims["LF"].mean(), color='red', linestyle='--', alpha=0.7)
    ax.set_xlabel("Laura Fernández (%)", fontsize=11)
    ax.set_ylabel("Álvaro Ramos (%)", fontsize=11)
    ax.set_title("Laura vs Segundo Lugar", fontweight='bold')
    ax.grid(alpha=0.3)

    # 6. Top-2 pares
    ax = axes[1, 2]
    top_pairs = out["top2_pairs"].head(8)
    pairs_labels = [f"{a.split()[0]} vs\n{b.split()[0]}" for a, b in top_pairs.index]
    ax.barh(range(len(top_pairs)), top_pairs.values*100, color='steelblue')
    ax.set_yticks(range(len(top_pairs)))
    ax.set_yticklabels(pairs_labels, fontsize=9)
    ax.set_xlabel("Probabilidad (%)", fontsize=11)
    ax.set_title("Top 8 combinaciones 1º vs 2º", fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig("simulacion_final_integrada.png", dpi=300, bbox_inches='tight')
    plt.close()

    print("✅ Visualización guardada: simulacion_final_integrada.png")

# =============================================================================
# REPORTE
# =============================================================================
def print_final_report(out: Dict) -> None:
    """Imprime reporte ejecutivo"""
    print("\n" + "="*80)
    print("📊 REPORTE FINAL - SIMULACIÓN INTEGRADA")
    print("="*80)

    summ = out["summary"]

    print(f"\nSimulaciones: {summ['n_sims']:,}")
    print(f"Fecha proyección: {summ['election_date']}")
    print(f"Encuestas usadas: {out['polls_used']} (papeleta/lista)")
    print(f"Ajustes históricos CR: {'✓ Aplicados' if out.get('historical_adjustments_applied') else '✗'}")

    print("\n" + "─"*80)
    print("🏆 LAURA FERNÁNDEZ - RESULTADO PROYECTADO")
    print("─"*80)

    print(f"\n  ✓ Probabilidad 1ª ronda (>40%): {summ['p_LF_win_round1']*100:.2f}%")
    print(f"  ✓ Probabilidad 1º lugar:        {summ['p_LF_top1']*100:.2f}%")

    print(f"\n  • Media:     {summ['LF_mean']:.2f}%")
    print(f"  • Mediana:   {summ['LF_median']:.2f}%")
    print(f"  • IC 90% (P5-P95):  [{summ['LF_p5']:.2f}% - {summ['LF_p95']:.2f}%]")
    print(f"  • IC 80% (P10-P90): [{summ['LF_p10']:.2f}% - {summ['LF_p90']:.2f}%]")

    print(f"\n  • Indecisos (U):    {summ['U_mean']:.2f}%")
    print(f"  • Blancos/nulos (B): {summ['B_mean']:.2f}%")

    print("\n" + "─"*80)
    print("🎯 INTERPRETACIÓN")
    print("─"*80)

    prob = summ['p_LF_win_round1'] * 100

    if prob >= 95:
        print("\n  🟢 VICTORIA EN PRIMERA RONDA: MUY ALTA PROBABILIDAD")
        print("     Laura Fernández supera el 40% en >95% de escenarios.")
    elif prob >= 85:
        print("\n  🟢 VICTORIA EN PRIMERA RONDA: ALTA PROBABILIDAD")
        print("     Laura Fernández supera el 40% en >85% de escenarios.")
    elif prob >= 70:
        print("\n  🟡 VICTORIA EN PRIMERA RONDA: PROBABLE")
        print("     Laura Fernández supera el 40% en >70% de escenarios.")
    elif prob >= 50:
        print("\n  🟡 ESCENARIO INCIERTO")
        print("     Probabilidad moderada de 1ª ronda, alta de 2ª ronda.")
    else:
        print("\n  🔴 SEGUNDA RONDA MUY PROBABLE")
        print("     Laura Fernández probablemente no supera el 40%.")

    print("\n" + "─"*80)
    print("📋 DATOS HISTÓRICOS INCORPORADOS")
    print("─"*80)
    print(f"\n  • Votantes habituales: {VOTANTES_HABITUALES*100:.0f}% (bajó de 47% en 1994)")
    print(f"  • Votantes ocasionales: {VOTANTES_OCASIONALES*100:.0f}% (aumentó)")
    print(f"  • Abstencionistas crónicos: {ABSTIENEN_SIEMPRE*100:.0f}% (muy bajo)")
    print(f"  • Participación esperada: ~60% (tendencia 2022)")

    print("\n" + "="*80)

# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    """Función principal"""
    print("\n" + "="*80)
    print("🗳️  SIMULACIÓN ELECTORAL COSTA RICA 2026 - VERSIÓN FINAL INTEGRADA")
    print("="*80)
    print("\nCombina:")
    print("  1. Modelado bayesiano WLS con logits")
    print("  2. Datos históricos abstencionismo CR (1982-2024)")
    print("  3. Patrones variabilidad electoral (votantes ocasionales)")
    print("  4. 12 encuestas octubre 2025 - enero 2026")
    print("  5. House effects + incertidumbre paramétrica")

    # Construir datos
    df_raw, polls = build_dataframe()
    df_wide = build_votes_wide(df_raw, polls)

    # Simular
    out = run_simulation(df_wide, n_sims=N_SIMS, seed=SEED)

    # Reporte
    print_final_report(out)

    # Guardar
    print("\n⏳ Guardando resultados...")
    out["sims"].to_csv("simulacion_final_integrada.csv", index=False)
    out["cand_summary"].to_csv("resumen_candidatos_final.csv")
    out["top2_pairs"].head(20).to_csv("top2_pairs_final.csv")

    # Visualizar
    create_comprehensive_plots(out)

    print("\n✅ SIMULACIÓN COMPLETADA")
    print("\nArchivos generados:")
    print("  • simulacion_final_integrada.csv (200,000 simulaciones)")
    print("  • resumen_candidatos_final.csv")
    print("  • top2_pairs_final.csv")
    print("  • simulacion_final_integrada.png")
    print("\n" + "="*80)

if __name__ == "__main__":
    main()
